# NUTDTS 816 Time Series Analysis
## L18 Foundation models

Lab notebook for Chapter 9 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 9.5 Foundation models for time series forecasting

In [ ]:
chronos_code = '''
import torch, pandas as pd, numpy as np
from chronos import BaseChronosPipeline
pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", device_map="cpu", torch_dtype=torch.float32)

def chronos_forecast(y_train, H, quantiles=(0.1, 0.5, 0.9)):
    """Zero-shot forecast: the last 512 points as context, returns the requested quantiles (H x len(quantiles))."""
    context = torch.tensor(y_train.values[-512:], dtype=torch.float32)
    qs, mean = pipe.predict_quantiles(context=[context], prediction_length=H, quantile_levels=list(quantiles))
    return qs[0].numpy(), mean[0].numpy()

# Point-forecast wrapper for rolling_origin (Chapter 7): median forecast
fit_forecast_chronos = lambda tr, h: chronos_forecast(tr, h)[0][:, 1]
# E, q = rolling_origin(dem, fit_forecast_chronos, h=14, first_origin=split, step=21, m=7)
'''

In [ ]:
timegpt_code = '''
from nixtla import NixtlaClient
client = NixtlaClient(api_key="YOUR_KEY")
df = pd.DataFrame({'unique_id': 'demand', 'ds': dem.index, 'y': dem.values})
fc = client.forecast(df=df, h=14, freq='D', level=[80, 95])      # point forecast with 80% and 95% intervals
'''

## Exercises

4. Run Chronos-Bolt zero-shot in Colab on the daily demand series and on the simulated monthly grid series; evaluate both on rolling origins against ETS. Where does it win, where does it lose, and what does the 80% interval coverage look like?
5. Explain to a health authority's data-protection officer what a foundation model API does with their case-count data, and what the alternative is.
6. Write the one-paragraph abstract of an MSc thesis that evaluates a fine-tuned foundation model against dynamic regression for Nigerian state-level inflation nowcasting.

In [ ]:
# Your work here
